# KTO: Socratic Style Alignment via Kahneman-Tversky Optimization

This notebook applies **KTO** (arXiv 2402.01306) to align the GSPO-trained model
toward Socratic tutoring behavior using unpaired preference data.

**Training pipeline stage:** 2 of 3 (GSPO (curriculum) → **KTO** → DPO)

**Target hardware:** Google Colab A100 80GB / H100 80GB / RTX 6000 Ada 96GB (auto-detect)

**Why KTO over DPO for this stage:**
- KTO works with **unpaired** preferences — each example is just good/bad, no need for matched pairs
- Perfect for our `dialogs.jsonl`: each assistant turn has a `move` type that naturally labels quality
- Based on prospect theory: losses loom larger than gains → model strongly avoids "telling" behavior
- Handles imbalanced data natively via `desirable_weight` / `undesirable_weight`

**Data sources:**
- `data/training/dialogs.jsonl` — 3,875 Socratic dialogues, 15,565 assistant turns
  - Good (label=True): scaffolding, encourage, hint, problematize moves
  - Bad (label=False): tell, rectify moves
- `training/data/preference_pairs.jsonl` — 12,597 preference pairs
  - Good: chosen responses (avg 4.1 guiding questions)
  - Bad: rejected responses (avg 0.9 questions, direct answers)

**Evaluation:** Removed from notebook — run locally via `training/scripts/evaluate_stage.py`

**Reference:** Ethayarajh et al. "KTO: Model Alignment as Prospect Theoretic Optimization" (arXiv 2402.01306)

In [ ]:
# Install dependencies
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# ============================================================
# ALL dependencies in one shot. After this cell: RESTART RUNTIME.
# After restart: SKIP this cell, start from Cell 2.
# ============================================================

# Step 1: Unsloth
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

# Step 2: Transformers v5 (Qwen3.5 hybrid arch) + ML
!pip install -q "transformers>=5.0.0" trl peft datasets

# Step 3: Fix Colab packages broken by numpy upgrade
!pip install -q --upgrade scipy torchvision "Pillow<12.0"

# Step 4: Other deps
!pip install -q accelerate bitsandbytes sentencepiece protobuf

print()
print("=" * 60)
print("  RESTART RUNTIME NOW: Runtime -> Restart session")
print("  After restart: SKIP this cell, run Cell 2 onwards.")
print("=" * 60)

In [ ]:
# ============================================================
# Mount Google Drive + add to sys.path
# ============================================================
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive")
print("Drive mounted, sys.path updated")

In [ ]:
# ============================================================
# Authenticate with HuggingFace Hub
# Required for: downloading gated models, pushing adapters
# ============================================================
from huggingface_hub import login

# Option 1: Interactive prompt (Colab)
login()

# Option 2: Use saved token / env variable (non-interactive)
# login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ============================================================
# Configuration
# ============================================================
import os
import torch
from datetime import datetime

BASE_MODEL = "Qwen/Qwen3.5-9B"

# GSPO adapter — HF repo (primary) + Drive fallback
GSPO_ADAPTER_HF    = "Siesher/mits-qwen3-9b-gspo"
GSPO_ADAPTER_LOCAL = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/final"

OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/kto_qwen3.5_9b"

# Data paths (on Drive)
DIALOGS_PATH    = "/content/drive/MyDrive/data/training/dialogs.jsonl"
PREFERENCE_PATH = "/content/drive/MyDrive/training/data/preference_pairs.jsonl"

# WandB
WANDB_PROJECT  = "mits-kto"
WANDB_RUN_NAME = f"kto-qwen3.5-9b-{datetime.now().strftime('%m%d-%H%M')}"

# ---- Auto-detect GPU: H100 / RTX 6000 (Ada/Blackwell) / A100 / Generic ----
_gpu_name   = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
_gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3) if torch.cuda.is_available() else 0

if "H100" in _gpu_name:
    GPU_TYPE = "H100"
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 32
elif "6000" in _gpu_name and _gpu_mem_gb >= 90:
    # Matches: RTX 6000 Ada, RTX PRO 6000 Blackwell Server Edition, etc.
    GPU_TYPE = f"RTX6000-96GB ({_gpu_name.split('NVIDIA ')[-1]})"
    BATCH_SIZE = 6
    GRADIENT_ACCUMULATION_STEPS = 5   # effective batch = 30
elif "A100" in _gpu_name and _gpu_mem_gb >= 70:
    GPU_TYPE = "A100-80GB"
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 8   # effective batch = 32
elif _gpu_mem_gb >= 40:
    GPU_TYPE = f"Generic ({_gpu_name})"
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 16
else:
    raise RuntimeError(
        f"GPU {_gpu_name} has only {_gpu_mem_gb:.0f}GB VRAM. "
        f"Disconnect and try again for a better GPU."
    )

print(f"Detected GPU: {_gpu_name} ({_gpu_mem_gb:.0f} GB) -> preset: {GPU_TYPE}")

# KTO hyperparameters (arXiv 2402.01306)
BETA                  = 0.1     # Prospect theory asymmetry (default, recommended)
LEARNING_RATE         = 5e-7    # Conservative for 9B model (paper: 5e-7 to 5e-6)
NUM_EPOCHS            = 2
MAX_LENGTH            = 1536    # prompt + completion total (TRL 0.27: single max_length)

# LoRA (merge on top of GSPO adapter)
LORA_R        = 16
LORA_ALPHA    = 32
MAX_SEQ_LENGTH = 2048

# Move classification for KTO labels
GOOD_MOVES = {"scaffolding", "encourage", "hint", "problematize", "problematising"}
BAD_MOVES  = {"tell", "rectify"}  # rectify = correcting but still telling

print(f"Base model:       {BASE_MODEL}")
print(f"GSPO adapter HF:  {GSPO_ADAPTER_HF}")
print(f"WandB run:        {WANDB_RUN_NAME}")
print(f"KTO beta={BETA}, LR={LEARNING_RATE}, epochs={NUM_EPOCHS}")
print(f"Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} effective")

In [ ]:
# ============================================================
# Load and prepare KTO dataset from two sources
# Source 1: dialogs.jsonl — multi-turn Socratic dialogues with move labels
# Source 2: preference_pairs.jsonl — chosen/rejected pairs → unpaired
# ============================================================
import json
from collections import Counter
from datasets import Dataset

kto_examples = []  # {prompt: [...messages], completion: [{role, content}], label: bool}


def classify_move(move: str) -> bool | None:
    """Classify a Socratic move as good (True), bad (False), or unknown (None)."""
    m = move.lower().strip()
    if m in GOOD_MOVES or m in BAD_MOVES:
        return m not in BAD_MOVES
    # Catch typos: problematize/problematise/problemize/etc. → good
    if m.startswith("problematiz") or m.startswith("problemat") or m.startswith("problemiz"):
        return True
    return None


# ---- Source 1: dialogs.jsonl ----
dialog_count = 0
move_labels = Counter()

dialogs_path = DIALOGS_PATH
if not os.path.exists(dialogs_path):
    dialogs_path = "/content/drive/MyDrive/MITS/data/training/dialogs.jsonl"

if os.path.exists(dialogs_path):
    with open(dialogs_path, "r", encoding="utf-8") as f:
        for line in f:
            dialog = json.loads(line)
            convs = dialog["conversations"]
            dialog_count += 1

            for i, turn in enumerate(convs):
                role = turn.get("role", turn.get("from", ""))
                content = turn.get("content", turn.get("value", ""))

                if role != "assistant":
                    continue

                try:
                    parsed = json.loads(content)
                    move = parsed.get("move", "unknown")
                    message = parsed.get("message", content)
                except (json.JSONDecodeError, TypeError):
                    continue

                label = classify_move(move)
                if label is None:
                    continue

                move_labels[f"{move}={'good' if label else 'bad'}"] += 1

                # Build prompt = all previous turns
                prompt_messages = []
                for j in range(i):
                    prev_role = convs[j].get("role", convs[j].get("from", ""))
                    prev_content = convs[j].get("content", convs[j].get("value", ""))
                    if prev_role == "assistant":
                        try:
                            prev_parsed = json.loads(prev_content)
                            prev_content = prev_parsed.get("message", prev_content)
                        except (json.JSONDecodeError, TypeError):
                            pass
                    prompt_messages.append({"role": prev_role, "content": prev_content})

                if not prompt_messages:
                    continue

                kto_examples.append({
                    "prompt": prompt_messages,
                    "completion": [{"role": "assistant", "content": message}],
                    "label": label,
                })

    print(f"Source 1 (dialogs): {dialog_count} dialogs → {sum(1 for e in kto_examples)} examples")
    print(f"  Move labels: {dict(move_labels.most_common())}")
else:
    print(f"WARNING: dialogs not found at {dialogs_path}")

dialog_examples = len(kto_examples)

# ---- Source 2: preference_pairs.jsonl ----
pref_path = PREFERENCE_PATH
if not os.path.exists(pref_path):
    pref_path = "/content/drive/MyDrive/MITS/training/data/preference_pairs.jsonl"

if os.path.exists(pref_path):
    pref_count = 0
    with open(pref_path, "r", encoding="utf-8") as f:
        for line in f:
            pair = json.loads(line)
            pref_count += 1
            prompt = pair["prompt"]

            if isinstance(prompt, str):
                prompt = [{"role": "user", "content": prompt}]

            chosen = pair["chosen"]
            rejected = pair["rejected"]

            if isinstance(chosen, str):
                chosen = [{"role": "assistant", "content": chosen}]
            elif isinstance(chosen, list) and isinstance(chosen[0], str):
                chosen = [{"role": "assistant", "content": chosen[0]}]

            if isinstance(rejected, str):
                rejected = [{"role": "assistant", "content": rejected}]
            elif isinstance(rejected, list) and isinstance(rejected[0], str):
                rejected = [{"role": "assistant", "content": rejected[0]}]

            kto_examples.append({"prompt": prompt, "completion": chosen, "label": True})
            kto_examples.append({"prompt": prompt, "completion": rejected, "label": False})

    print(f"Source 2 (preference_pairs): {pref_count} pairs → {(len(kto_examples) - dialog_examples)} examples")
else:
    print(f"WARNING: preference_pairs not found at {pref_path}")

# ---- Stats ----
good_count = sum(1 for e in kto_examples if e["label"])
bad_count = sum(1 for e in kto_examples if not e["label"])
print(f"\nTotal KTO examples: {len(kto_examples)}")
print(f"  Good (label=True): {good_count} ({100*good_count/len(kto_examples):.1f}%)")
print(f"  Bad (label=False): {bad_count} ({100*bad_count/len(kto_examples):.1f}%)")

# Compute balancing weights (KTO paper: ratio should be 1:1 to 4:3)
if good_count > bad_count:
    DESIRABLE_WEIGHT = 1.0
    UNDESIRABLE_WEIGHT = min(good_count / bad_count, 4/3)
else:
    UNDESIRABLE_WEIGHT = 1.0
    DESIRABLE_WEIGHT = min(bad_count / good_count, 4/3)

print(f"  Balancing: desirable_weight={DESIRABLE_WEIGHT:.3f}, undesirable_weight={UNDESIRABLE_WEIGHT:.3f}")
print(f"  Effective ratio: {DESIRABLE_WEIGHT * good_count:.0f} : {UNDESIRABLE_WEIGHT * bad_count:.0f}")

In [ ]:
# ============================================================
# Build HuggingFace Dataset for KTOTrainer
# Format: {prompt: [messages], completion: [messages], label: bool}
# ============================================================
import random

random.seed(42)
random.shuffle(kto_examples)

# Train/eval split (5% eval)
eval_size = max(100, int(0.05 * len(kto_examples)))
train_examples = kto_examples[eval_size:]
eval_examples = kto_examples[:eval_size]

train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples)

print(f"Train: {len(train_dataset)} examples")
print(f"  Good: {sum(1 for e in train_examples if e['label'])}, Bad: {sum(1 for e in train_examples if not e['label'])}")
print(f"Eval: {len(eval_dataset)} examples")
print(f"  Good: {sum(1 for e in eval_examples if e['label'])}, Bad: {sum(1 for e in eval_examples if not e['label'])}")

In [ ]:
# ============================================================
# Load GSPO-trained model + apply fresh LoRA for KTO
# Strategy: load base model, merge GSPO adapter, add new LoRA
# ============================================================
import torch
from unsloth import FastLanguageModel

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

# Load GSPO adapter — try HF first, fallback to local Drive path
GSPO_SOURCE = None
from peft import PeftModel

try:
    model_gspo = PeftModel.from_pretrained(model, GSPO_ADAPTER_HF)
    model = model_gspo.merge_and_unload()
    GSPO_SOURCE = GSPO_ADAPTER_HF
    print(f"Merged GSPO adapter from HuggingFace: {GSPO_ADAPTER_HF}")
except Exception as _e_hf:
    print(f"HF load failed ({_e_hf}), trying local Drive...")
    if os.path.exists(GSPO_ADAPTER_LOCAL):
        model = PeftModel.from_pretrained(model, GSPO_ADAPTER_LOCAL).merge_and_unload()
        GSPO_SOURCE = GSPO_ADAPTER_LOCAL
        print(f"Merged GSPO adapter from Drive: {GSPO_ADAPTER_LOCAL}")
    else:
        print("WARNING: GSPO adapter not found (HF or local). Training KTO on base model.")
        GSPO_SOURCE = "base_only"

# Apply fresh LoRA for KTO
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

# Qwen3.5 returns Processor (multimodal), unwrap to tokenizer
if not hasattr(tokenizer, "vocab_size") and hasattr(tokenizer, "tokenizer"):
    _processor = tokenizer
    tokenizer  = _processor.tokenizer
    print(f"Unwrapped Qwen3VLProcessor -> {type(tokenizer).__name__}")
print(f"GSPO source: {GSPO_SOURCE}")
print(f"Fresh LoRA for KTO: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Qwen3.5 text-only fix: replace compute_3d_position_ids entirely
# NOTE: **kwargs required — transformers v5 passes mm_token_type_ids, cache_position, etc.
import torch as _torch
_qwen_inner = model
while hasattr(_qwen_inner, "model"):
    _qwen_inner = _qwen_inner.model

def _text_only_pos_ids(self, input_ids=None, inputs_embeds=None,
                        image_grid_thw=None, video_grid_thw=None,
                        attention_mask=None, past_key_values=None,
                        **kwargs):
    if inputs_embeds is not None:
        bs, sl = inputs_embeds.shape[:2]
        dev = inputs_embeds.device
    else:
        bs, sl = input_ids.shape
        dev = input_ids.device
    past = 0
    if past_key_values is not None and hasattr(past_key_values, "get_seq_length"):
        try: past = past_key_values.get_seq_length()
        except: past = 0
    if attention_mask is not None and past == 0:
        pos = attention_mask.long().cumsum(-1) - 1
        pos.masked_fill_(attention_mask == 0, 1)
        pos = pos.unsqueeze(0).expand(3, -1, -1)
    else:
        pos = _torch.arange(past, past + sl, device=dev).view(1, 1, -1).expand(3, bs, -1)
    self.rope_deltas = _torch.zeros(bs, 1, dtype=_torch.long, device=dev)
    return pos

type(_qwen_inner).compute_3d_position_ids = _text_only_pos_ids
print(f"compute_3d_position_ids replaced (text-only, **kwargs-safe) on {type(_qwen_inner).__name__}")

In [ ]:
# ============================================================
# KTO Training (arXiv 2402.01306)
# Prospect-theoretic alignment: losses loom larger than gains
# ============================================================
# Fix: llm_blender uses removed TRANSFORMERS_CACHE (transformers v5)
import transformers.utils.hub as _tf_hub
if not hasattr(_tf_hub, "TRANSFORMERS_CACHE"):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        _tf_hub.TRANSFORMERS_CACHE = HF_HUB_CACHE
    except ImportError:
        _tf_hub.TRANSFORMERS_CACHE = "/root/.cache/huggingface/hub"

import wandb
from trl import KTOTrainer, KTOConfig

wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    tags=["kto", "qwen3.5-9b", "socratic", GPU_TYPE.lower().replace(" ", "-")],
    config={
        "stage": "kto",
        "base_model": BASE_MODEL,
        "gspo_source": GSPO_SOURCE,
        "beta": BETA,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "desirable_weight": DESIRABLE_WEIGHT,
        "undesirable_weight": UNDESIRABLE_WEIGHT,
        "gpu_type": GPU_TYPE,
    },
)

training_args = KTOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=BETA,
    desirable_weight=DESIRABLE_WEIGHT,
    undesirable_weight=UNDESIRABLE_WEIGHT,
    max_length=MAX_LENGTH,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=100,
    optim="adamw_torch_fused",
    max_grad_norm=1.0,
    seed=42,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
)

trainer = KTOTrainer(
    model=model,
    ref_model=None,       # Uses implicit reference (PEFT)
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print(f"KTO Training config:")
print(f"  beta={BETA} (prospect theory asymmetry)")
print(f"  desirable_weight={DESIRABLE_WEIGHT:.3f}, undesirable_weight={UNDESIRABLE_WEIGHT:.3f}")
print(f"  LR={LEARNING_RATE}, epochs={NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  WandB: {WANDB_PROJECT}/{WANDB_RUN_NAME}")
print(f"Starting training...")

result = trainer.train()

print(f"\nKTO training complete!")
print(f"  Final loss: {result.training_loss:.4f}")
print(f"  Metrics: {result.metrics}")

In [ ]:
# ============================================================
# Save KTO adapter + push to HuggingFace
# ============================================================
import json

final_path = os.path.join(OUTPUT_DIR, "final")
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"KTO adapter saved to {final_path}")

# Save training metadata
meta = {
    "stage": "kto",
    "base_model": BASE_MODEL,
    "gspo_source": GSPO_SOURCE,
    "beta": BETA,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "lora_r": LORA_R,
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
    "desirable_weight": DESIRABLE_WEIGHT,
    "undesirable_weight": UNDESIRABLE_WEIGHT,
    "final_loss": result.training_loss,
    "data_sources": ["dialogs.jsonl", "preference_pairs.jsonl"],
}
with open(os.path.join(final_path, "kto_training_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

# Log final metrics to WandB and close run
wandb.log({f"final/{k}": v for k, v in result.metrics.items()})
wandb.log({"final/loss": result.training_loss, "final/gspo_source": GSPO_SOURCE})
wandb.finish()
print("WandB run finished")

# Optional: push to HuggingFace Hub
HF_PUSH = False  # Set True to push
if HF_PUSH:
    model.push_to_hub("Siesher/mits-qwen3-9b-kto")
    tokenizer.push_to_hub("Siesher/mits-qwen3-9b-kto")
    print("Pushed to Siesher/mits-qwen3-9b-kto")

In [ ]:
# ============================================================
# Evaluation removed — run locally:
#   python training/scripts/evaluate_stage.py --stage kto \
#     --adapter /content/drive/MyDrive/MITS/checkpoints/kto_qwen3.5_9b/final
# ============================================================
print("Socratic evaluation skipped (run locally)")